In [1]:
import pandas as pd

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Read in the csv file as a dataframe using pandas

df = pd.read_csv("quiz4_invest.csv", index_col='Project')

# read the parameters
Capital = {}
Sector = {}
Risk = {}
Duration = {}
ExpectedReturn = {}

for i in df.index:  # iterate over the row index
    Capital[i] = float(df['Capital ($M)'][i])
    Sector[i] = df['Sector'][i]
    Risk[i] = df['Risk'][i]
    Duration[i] = float(df['Duration'][i])
    ExpectedReturn[i] = float(df['Expected return ($M)'][i])
    
Projects = Capital.keys()
Budget = 100

In [3]:
Capital

{'Project 1': 20.0,
 'Project 2': 15.0,
 'Project 3': 23.0,
 'Project 4': 10.0,
 'Project 5': 20.0,
 'Project 6': 12.0,
 'Project 7': 20.0,
 'Project 8': 25.0,
 'Project 9': 14.0,
 'Project 10': 10.0}

In [4]:
ExpectedReturn

{'Project 1': 34.0,
 'Project 2': 36.0,
 'Project 3': 50.0,
 'Project 4': 16.0,
 'Project 5': 54.0,
 'Project 6': 32.0,
 'Project 7': 60.0,
 'Project 8': 48.0,
 'Project 9': 38.0,
 'Project 10': 12.0}

In [5]:
from docplex.mp.model import Model
mdl = Model()

In [6]:
# variables
select = mdl.binary_var_dict(Projects, name='Select')

In [7]:
# objective
mdl.minimize(mdl.sum(Capital[i]*select[i] for i in select))

In [8]:
# constraint - No more than 100$M can be invested in total

MaxInvested = mdl.add_constraint(mdl.sum(Capital[i]*select[i] for i in select) <= Budget)


In [9]:
# constraint - At most two projects in IT can be selected

mdl.add_constraint(mdl.sum(select[i] for i in Projects if Sector[i]=='IT') <= 2)


docplex.mp.LinearConstraint[](Select_Project 1+Select_Project 6+Select_Project 10,LE,2)

In [10]:
# constraint - At least one Healthcare project must be selected

mdl.add_constraint(mdl.sum(select[i] for i in Projects if Sector[i]=='Healthcare') >= 1)

docplex.mp.LinearConstraint[](Select_Project 2+Select_Project 3,GE,1)

In [11]:
# constraint - The number of projects with Low risk must be at least the number of projects with High risk

LowRiskProjects = mdl.sum(select[i] for i in Projects if Risk[i]=='Low')
HighRiskProjects = mdl.sum(select[i] for i in Projects if Risk[i]=='High')
mdl.add_constraint(LowRiskProjects >= HighRiskProjects)

docplex.mp.LinearConstraint[](Select_Project 1+Select_Project 8+Select_Project 10,GE,Select_Project 3+Select_Project 6+Select_Project 7)

In [12]:
# constraint - At least three projects with a duration of 4 years or shorter must be selected

mdl.add_constraint(mdl.sum(select[i] for i in Projects if Duration[i] <= 4) >= 3)

docplex.mp.LinearConstraint[](Select_Project 1+Select_Project 2+Select_Project 3+Select_Project 4+Select_Project 6+Select_Project 9+Select_Project 10,GE,3)

In [13]:
# NEW constraint - expected return of at least $200 million.

ExpReturn = mdl.add_constraint(mdl.sum(ExpectedReturn[i]*select[i] for i in select) >= 200)

In [14]:
ExpReturn

docplex.mp.LinearConstraint[](34Select_Project 1+36Select_Project 2+50Select_Project 3+16Select_Project 4+54Select_Project 5+32Select_Project 6+60Select_Project 7+48Select_Project 8+38Select_Project 9+12Select_Project 10,GE,200)

In [15]:
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.010318,status='integer optimal solution')

In [16]:
mdl.print_solution()

objective: 79.000
status: OPTIMAL_SOLUTION(2)
  "Select_Project 2"=1
  "Select_Project 5"=1
  "Select_Project 7"=1
  "Select_Project 9"=1
  "Select_Project 10"=1
